In [ ]:
import pickle
import shutil
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import kagglehub

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

BASE_DIR    = Path().resolve().parent
DATA_DIR    = BASE_DIR / 'data'
MODELS_DIR  = BASE_DIR / 'models'
FIGURES_DIR = BASE_DIR / 'figures'
TREC_DIR    = MODELS_DIR / 'trec'

DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
TREC_DIR.mkdir(exist_ok=True)

print(f'Projeto em  : {BASE_DIR}')
print(f'Dados em    : {DATA_DIR}')
print(f'Modelos em  : {TREC_DIR}')
print(f'Figuras em  : {FIGURES_DIR}')

In [ ]:
ENRON_DIR   = MODELS_DIR / 'enron_original'
LEGACY_DIRS = [MODELS_DIR / 'exp0_enron_original']
CANDIDATES  = [
    'X_train_tfidf.pkl', 'X_test_tfidf.pkl',
    'y_train.pkl', 'y_test.pkl', 'tfidf.pkl',
    'model_naive_bayes.pkl', 'model_svm.pkl',
    'model_random_forest.pkl', 'model_regressao.pkl',
    'results.csv',
]

ENRON_DIR.mkdir(exist_ok=True)
movidos = []
ja_existiam = []

for fname in CANDIDATES:
    dst = ENRON_DIR / fname
    src_root = MODELS_DIR / fname
    if src_root.exists() and not dst.exists():
        shutil.copy2(str(src_root), str(dst))
        src_root.unlink()
        movidos.append(f'models/{fname} → enron_original/{fname}')
        continue
    if not dst.exists():
        for legacy in LEGACY_DIRS:
            src_leg = legacy / fname
            if src_leg.exists():
                shutil.copy2(str(src_leg), str(dst))
                movidos.append(f'{legacy.name}/{fname} → enron_original/{fname}')
                break
    else:
        ja_existiam.append(fname)

print(f'Pasta criada: {ENRON_DIR}')
if movidos:
    print('\nArquivos preservados:')
    for m in movidos:
        print(f'  ✓ {m}')
if ja_existiam:
    print('\nJá existiam em enron_original/ (não sobrescritos):')
    for f in ja_existiam:
        print(f'  • {f}')
if not movidos and not ja_existiam:
    print('Nenhum arquivo Enron encontrado para mover.')

print('\nConteúdo de models/enron_original/:')
for p in sorted(ENRON_DIR.iterdir()):
    print(f'  {p.name}')

In [ ]:
TREC_CSV = DATA_DIR / 'trec_2007.csv'

if not TREC_CSV.exists():
    print('Baixando dataset via kagglehub...')
    kaggle_path = kagglehub.dataset_download(
        'imdeepmind/preprocessed-trec-2007-public-corpus-dataset'
    )
    print(f'Path do cache kagglehub: {kaggle_path}')

    csv_files = list(Path(kaggle_path).rglob('*.csv'))
    if not csv_files:
        raise FileNotFoundError(f'Nenhum CSV encontrado em {kaggle_path}')
    print(f'Arquivos CSV encontrados: {[f.name for f in csv_files]}')

    # Maior CSV = dataset principal (exclui arquivos auxiliares menores)
    src = max(csv_files, key=lambda f: f.stat().st_size)
    shutil.copy2(str(src), str(TREC_CSV))
    print(f'Copiado: {src.name} → {TREC_CSV}')
else:
    print(f'Dataset já existe: {TREC_CSV}')

df_raw = pd.read_csv(TREC_CSV, encoding='latin-1', on_bad_lines='skip')

print(f'\nShape: {df_raw.shape}')
print(f'Colunas: {df_raw.columns.tolist()}')
print(f'\nPrimeiras linhas:')
print(df_raw.head(3).to_string())

TEXT_CANDIDATES  = ['body', 'message', 'text', 'email', 'content', 'mail']
LABEL_CANDIDATES = ['label', 'spam', 'class', 'category', 'spam/ham', 'type', 'tag']

cols_lower = {c.lower(): c for c in df_raw.columns}

text_col  = next((cols_lower[c] for c in TEXT_CANDIDATES  if c in cols_lower), None)
label_col = next((cols_lower[c] for c in LABEL_CANDIDATES if c in cols_lower), None)

# Fallback por heurística: maior coluna string = texto; menor cardinalidade = rótulo
if text_col is None:
    str_cols = df_raw.select_dtypes(include='object').columns.tolist()
    text_col = max(str_cols, key=lambda c: df_raw[c].str.len().mean())

if label_col is None:
    str_cols = df_raw.select_dtypes(include='object').columns.tolist()
    label_col = min([c for c in str_cols if c != text_col],
                    key=lambda c: df_raw[c].nunique())

print(f'\nColuna de texto detectada : "{text_col}"')
print(f'Coluna de rótulo detectada: "{label_col}"')
print(f'\nDistribuição de classes:')
print(df_raw[label_col].value_counts())

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

def truncate_to_150_words(text):
    words = str(text).split()
    return ' '.join(words[:150])

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    return ' '.join(tokens)

def full_preprocess(text):
    return preprocess(truncate_to_150_words(text))

exemplo = 'Hello! This is a TEST email. Buy now: free offer limited time 100% guaranteed!!!'
print('Original  :', exemplo)
print('Truncado  :', truncate_to_150_words(exemplo))
print('Processado:', preprocess(truncate_to_150_words(exemplo)))

In [ ]:
df = df_raw[[text_col, label_col]].copy()
df = df.dropna(subset=[text_col])

print(f'E-mails após remover nulos: {len(df)}')
print('Aplicando pré-processamento (pode levar alguns instantes)...')

df['text_processed']  = df[text_col].apply(full_preprocess)
df['n_words']         = df['text_processed'].str.split().str.len()
df['n_words_truncated'] = df[text_col].apply(lambda x: len(truncate_to_150_words(x).split()))

print(f'Concluído. Shape: {df.shape}')
print('\nExemplo:')
print(f'  Original  : {str(df[text_col].iloc[0])[:120]}...')
print(f'  Processado: {df["text_processed"].iloc[0][:120]}...')

In [ ]:
antes = len(df)
df = df[df['text_processed'].str.strip() != '']
print(f'Removidos {antes - len(df)} e-mails vazios após pré-processamento.')

valores_unicos = df[label_col].dropna().unique()
print(f'\nValores únicos na coluna de rótulo: {valores_unicos}')

label_map = {}
for v in valores_unicos:
    v_lower = str(v).strip().lower()
    if v_lower in ('spam', '1', 'yes', 'true', 's'):
        label_map[v] = 1
    elif v_lower in ('ham', '0', 'no', 'false', 'h', 'legitimate'):
        label_map[v] = 0

# Fallback: ordem alfabética — menor valor = ham (0)
if len(label_map) != len(valores_unicos):
    sorted_vals = sorted([str(v) for v in valores_unicos])
    label_map = {v: i for i, v in enumerate(sorted_vals)}
    print(f'Mapeamento por ordem alfabética: {label_map}')
else:
    print(f'Mapeamento aplicado: {label_map}')

df['label'] = df[label_col].map(label_map)
df = df[df['label'].notna()]
df['label'] = df['label'].astype(int)

print(f'\nCorpus final: {len(df)} e-mails')
print(df['label'].value_counts().rename({1: 'spam', 0: 'ham'}))

In [ ]:
total   = len(df)
n_spam  = (df['label'] == 1).sum()
n_ham   = (df['label'] == 0).sum()
pct_150 = (df['n_words_truncated'] >= 150).mean()

print('=' * 55)
print('ESTATÍSTICAS DO CORPUS FINAL — TREC 2007')
print('=' * 55)
print(f'Total de e-mails utilizáveis : {total:>8,}')
print(f'  Spam (1)                   : {n_spam:>8,}  ({n_spam/total:.1%})')
print(f'  Ham  (0)                   : {n_ham:>8,}  ({n_ham/total:.1%})')
print(f'\nE-mails com ≥ 150 palavras   : {pct_150:.1%}')
print(f'(para estes, o recorte é exato; para os demais, usa o texto integral)')
print('\nComprimento em palavras (após truncagem e processamento):')
print(df['n_words'].describe().round(1).to_string())
print('\nComprimento truncado bruto (antes do processamento NLP):')
print(df['n_words_truncated'].describe().round(1).to_string())
print('=' * 55)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    'Análise Exploratória — TREC 2007 Public Spam Corpus',
    fontsize=13, fontweight='bold'
)

ax1 = axes[0]
comprimentos = df['n_words_truncated']
mediana = comprimentos.median()

ax1.hist(comprimentos, bins=40, color='#2563EB', edgecolor='white', alpha=0.85)
ax1.axvline(150,     color='red',    linewidth=1.8, linestyle='--',
            label=f'Recorte: 150 palavras')
ax1.axvline(mediana, color='orange', linewidth=1.8, linestyle='-.',
            label=f'Mediana: {mediana:.0f} palavras')
ax1.set_title('Distribuição do Comprimento dos E-mails', fontweight='bold')
ax1.set_xlabel('Número de palavras (até 150)')
ax1.set_ylabel('Frequência')
ax1.legend(fontsize=9)

ax2 = axes[1]
contagens = df['label'].value_counts().sort_index()
cores     = ['#1C2B4A', '#0891B2']

bars = ax2.bar(['Ham (0)', 'Spam (1)'], contagens.values, color=cores, edgecolor='white', width=0.5)
for bar, val in zip(bars, contagens.values):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + contagens.max() * 0.01,
        f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
ax2.set_title('Distribuição de Classes', fontweight='bold')
ax2.set_ylabel('Número de e-mails')
ax2.set_ylim(0, contagens.max() * 1.12)

fig.text(0.5, -0.03, 'Fonte: Elaborado pelo autor.', ha='center', fontsize=9, style='italic')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'exploratory_analysis_trec.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Figura salva: {FIGURES_DIR / "exploratory_analysis_trec.png"}')

In [ ]:
X = df['text_processed']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Treino : {len(X_train):,} e-mails')
print(f'Teste  : {len(X_test):,} e-mails')
print(f'\nProporção spam no treino: {y_train.mean():.2%}')
print(f'Proporção spam no teste : {y_test.mean():.2%}')

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    sublinear_tf=True,
    min_df=2,
    ngram_range=(1, 2)
)

# fit apenas no treino — evita data leakage no conjunto de teste
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Shape treino : {X_train_tfidf.shape}')
print(f'Shape teste  : {X_test_tfidf.shape}')
print(f'Vocabulário  : {len(tfidf.vocabulary_):,} termos')

print('\nTop 20 termos por TF-IDF médio no treino:')
scores    = X_train_tfidf.mean(axis=0).A1
top_idx   = scores.argsort()[-20:][::-1]
vocab_inv = {v: k for k, v in tfidf.vocabulary_.items()}
for idx in top_idx:
    print(f'  {vocab_inv[idx]:<30} {scores[idx]:.4f}')

In [ ]:
artefatos = {
    'X_train_tfidf': X_train_tfidf,
    'X_test_tfidf':  X_test_tfidf,
    'y_train':       y_train,
    'y_test':        y_test,
    'tfidf':         tfidf,
}

for nome, obj in artefatos.items():
    caminho = TREC_DIR / f'{nome}.pkl'
    with open(caminho, 'wb') as f:
        pickle.dump(obj, f)
    print(f'  Salvo: {caminho}')

print('\n' + '=' * 55)
print('RESUMO — ARQUIVOS GERADOS')
print('=' * 55)
print(f'Figura  : {FIGURES_DIR / "exploratory_analysis_trec.png"}')
for nome in artefatos:
    print(f'Artefato: {TREC_DIR / (nome + ".pkl")}')
print('=' * 55)